# Midati — Market product to recipe-ingredient review

This notebook builds a **review queue**, not an automatic approval system.

It downloads the market catalogue and canonical ingredient catalogue from one immutable Git commit, validates the source files and identity crosswalk, creates a multilingual candidate CSV, and exports the JSON format already understood by Midati's Android Room installer.

**Safety rules**
- Text similarity only creates `SUGGESTED` candidates.
- A mapping becomes `VERIFIED` only when a reviewer explicitly writes `APPROVE` in the CSV.
- Products flagged for food-scope review cannot be approved here.
- Different product forms (for example canned vs dried, fresh vs frozen, powder vs paste) and incompatible package/ingredient unit families are not matched.
- When uncertain, leave the row blank or reject it. Unmatched products remain without a price mapping.

No source data upload or GitHub token is required. The notebook writes local output files for review and later commit to the feature branch.

## Run order

1. Run the setup and candidate-generation cells from the top through **Generate the review queue**. The notebook downloads and validates the pinned catalogue, runs the matcher unit tests, and downloads `ingredient-match-review.csv`.
2. Open that CSV in Excel, LibreOffice, or Google Sheets. Change only `review_decision` and (optionally) `review_note`; preserve every row and its IDs. Approve at most one candidate per product.
3. Return to Colab and run the upload cell. Upload exactly one reviewed CSV. Then run the export cell to download `ingredient-matches.json` and `midati-product-ingredient-review-bundle.zip`.
4. Add the reviewed `ingredient-matches.json` to this repository at `data/market-products/ingredient-matches.json` on `main`. Do not replace product parts or edit the manifest.

Do not use **Run all** before reviewing the CSV: the upload cell is intentionally a separate review checkpoint.

In [1]:
from pathlib import Path
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError
from datetime import datetime, timezone
from collections import Counter
import csv
import hashlib
import json
import re
import subprocess
import sys
import time
import zipfile

from google.colab import files

REPOSITORY = "ni3yyn/Meals"
SOURCE_COMMIT = "203e0174db7241738dd75a1764eb80a88fca6a4b"
EXPECTED_ROOM_SCHEMA = 19
EXPECTED_MARKET_PRODUCTS = 1803
EXPECTED_FOOD_ITEMS = 4649
EXPECTED_CROSSWALK_KEYS = 1261
EXPECTED_ELIGIBLE_INGREDIENTS = 838

WORK_DIR = Path("/content/midati-product-ingredient-review")
SOURCE_DIR = WORK_DIR / "pipeline-source"
OUTPUT_DIR = WORK_DIR / "output"
WORK_DIR.mkdir(parents=True, exist_ok=True)
SOURCE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def fetch_bytes(url: str, attempts: int = 4) -> bytes:
    request = Request(
        url,
        headers={"User-Agent": "Midati-Colab-Matching-Pipeline/1.0"},
    )
    last_error = None
    for attempt in range(attempts):
        try:
            with urlopen(request, timeout=90) as response:
                return response.read()
        except (HTTPError, URLError, TimeoutError) as error:
            last_error = error
            if attempt + 1 < attempts:
                time.sleep(2 ** attempt)
    raise RuntimeError(f"Unable to fetch pinned source after {attempts} attempts: {url}") from last_error

def raw_url(path: str) -> str:
    return f"https://raw.githubusercontent.com/{REPOSITORY}/{SOURCE_COMMIT}/{path}"

def load_json_bytes(data: bytes, label: str):
    try:
        return json.loads(data.decode("utf-8"))
    except Exception as error:
        raise ValueError(f"Invalid UTF-8 JSON at {label}") from error

print("Pinned source commit:", SOURCE_COMMIT)
tree_url = f"https://api.github.com/repos/{REPOSITORY}/git/trees/{SOURCE_COMMIT}?recursive=1"
tree_payload = load_json_bytes(fetch_bytes(tree_url), "Git tree")
tree_entries = tree_payload.get("tree", [])
if tree_payload.get("truncated"):
    raise RuntimeError("GitHub returned a truncated repository tree; refusing a partial source snapshot.")

market_paths = sorted(
    item["path"]
    for item in tree_entries
    if item.get("type") == "blob"
    and item["path"].startswith("data/market-products/")
    and item["path"].endswith(".json")
    and item["path"].rsplit("/", 1)[-1] != "ingredient-matches.json"
)
if not market_paths:
    raise RuntimeError("No pinned market-product JSON files were found.")

market_bytes = {}
for index, path in enumerate(market_paths, start=1):
    market_bytes[path] = fetch_bytes(raw_url(path))
    print(f"Downloaded market source {index}/{len(market_paths)}: {path}")

# This is byte-for-byte equivalent to MARKET_PRODUCT_BASE_CATALOGUE_REVISION
# in app/build.gradle.kts: relative path, NUL, file bytes, NUL, sorted by path.
market_digest = hashlib.sha256()
for path in market_paths:
    relative_path = path.removeprefix("data/market-products/")
    market_digest.update(relative_path.encode("utf-8"))
    market_digest.update(b"\0")
    market_digest.update(market_bytes[path])
    market_digest.update(b"\0")
SOURCE_CATALOGUE_REVISION = market_digest.hexdigest()

manifest = load_json_bytes(
    market_bytes["data/market-products/manifest.json"],
    "data/market-products/manifest.json",
)
if manifest.get("target", {}).get("repository") != REPOSITORY:
    raise ValueError("Market catalogue belongs to a different repository.")
if int(manifest.get("target", {}).get("roomSchemaVersion", -1)) != EXPECTED_ROOM_SCHEMA:
    raise ValueError("Market catalogue Room schema version does not match this pipeline.")
if int(manifest.get("counts", {}).get("products", -1)) != EXPECTED_MARKET_PRODUCTS:
    raise ValueError("Pinned market product count changed; review the source pin before continuing.")

market_parts = []
for relative_path in manifest.get("parts", []):
    full_path = "data/" + relative_path
    if full_path not in market_bytes:
        raise FileNotFoundError(f"Manifest references a missing part: {relative_path}")
    market_parts.append(load_json_bytes(market_bytes[full_path], full_path))

# Download the canonical ingredient catalogue, the legacy-to-canonical crosswalk,
# the seed-ID declarations, and the exact pipeline/tests from this same commit.
required_paths = {
    "data/products/catalogue.json": None,
    "data/products/recipe-ingredient-map.json": None,
    "app/src/main/java/dz/meals/app/data/FoodIdentityResolver.kt": None,
    "tools/market_product_matching/pipeline.py": None,
    "tools/market_product_matching/test_pipeline.py": None,
}
for path in required_paths:
    required_paths[path] = fetch_bytes(raw_url(path))

food_catalogue = load_json_bytes(required_paths["data/products/catalogue.json"], "food catalogue")
identity_crosswalk = load_json_bytes(required_paths["data/products/recipe-ingredient-map.json"], "recipe ingredient crosswalk")
food_items = food_catalogue.get("products", [])
legacy_to_food = identity_crosswalk.get("legacyRoomIngredientIdToFoodItemId", {})
if len(food_items) != EXPECTED_FOOD_ITEMS:
    raise ValueError("Canonical food catalogue count changed; review the source pin.")
if len(legacy_to_food) != EXPECTED_CROSSWALK_KEYS:
    raise ValueError("Canonical identity crosswalk count changed; review the source pin.")

resolver_source = required_paths["app/src/main/java/dz/meals/app/data/FoodIdentityResolver.kt"].decode("utf-8")
seed_food_ids = set(re.findall(r'"[^"]+"\s+to\s+"([0-9a-fA-F-]{36})"', resolver_source))
eligible_food_ids = set(legacy_to_food.values()) | seed_food_ids
food_by_id = {str(item.get("id", "")).lower(): item for item in food_items}
missing_food_ids = sorted(eligible_food_ids - set(food_by_id))
if missing_food_ids:
    raise ValueError(f"Identity crosswalk references food IDs absent from the canonical catalogue: {missing_food_ids[:10]}")
eligible_food_items = [food_by_id[food_id] for food_id in sorted(eligible_food_ids)]
if len(eligible_food_items) != EXPECTED_ELIGIBLE_INGREDIENTS:
    raise ValueError(
        f"Expected {EXPECTED_ELIGIBLE_INGREDIENTS} recipe/seed-backed ingredient identities, "
        f"found {len(eligible_food_items)}. Do not silently broaden the target set."
    )

# Compute FOOD_CATALOGUE_REVISION exactly as app/build.gradle.kts does.
food_digest = hashlib.sha256()
for index, path in enumerate((
    "data/products/catalogue.json",
    "data/products/categories.json",
)):
    body = required_paths.get(path)
    if body is None:
        body = fetch_bytes(raw_url(path))
    food_digest.update(str(index).encode("utf-8"))
    food_digest.update(b"\0")
    food_digest.update(body)
    food_digest.update(b"\0")
FOOD_CATALOGUE_REVISION = food_digest.hexdigest()

print("Pinned market base revision:", SOURCE_CATALOGUE_REVISION)
print("Pinned food catalogue revision:", FOOD_CATALOGUE_REVISION)
print("Canonical food items:", len(food_items))
print("Eligible recipe/seed-backed ingredients:", len(eligible_food_items))
print("Seed IDs:", len(seed_food_ids))
print("Crosswalk legacy IDs:", len(legacy_to_food))


Pinned source commit: 203e0174db7241738dd75a1764eb80a88fca6a4b
Downloaded market source 1/10: data/market-products/build-audit.json
Downloaded market source 2/10: data/market-products/identity_registry.json
Downloaded market source 3/10: data/market-products/manifest.json
Downloaded market source 4/10: data/market-products/parts/market-products-001.json
Downloaded market source 5/10: data/market-products/parts/market-products-002.json
Downloaded market source 6/10: data/market-products/parts/market-products-003.json
Downloaded market source 7/10: data/market-products/parts/market-products-004.json
Downloaded market source 8/10: data/market-products/parts/market-products-005.json
Downloaded market source 9/10: data/market-products/parts/market-products-006.json
Downloaded market source 10/10: data/market-products/parts/market-products-007.json
Pinned market base revision: 9980680f1cfc920efe1e0bbc592a272cd5b247acc3d179b43d968d48c17d5901
Pinned food catalogue revision: 29cb3e292f2866f99d1

In [2]:
from importlib.util import module_from_spec, spec_from_file_location

# Materialize the pipeline and its unit tests under a small isolated source tree.
package_dir = SOURCE_DIR / "tools" / "market_product_matching"
package_dir.mkdir(parents=True, exist_ok=True)
(SOURCE_DIR / "tools" / "__init__.py").write_text("", encoding="utf-8")
package_dir.joinpath("__init__.py").write_text("", encoding="utf-8")
(package_dir / "pipeline.py").write_bytes(required_paths["tools/market_product_matching/pipeline.py"])
(package_dir / "test_pipeline.py").write_bytes(required_paths["tools/market_product_matching/test_pipeline.py"])

sys.path.insert(0, str(SOURCE_DIR))
test_run = subprocess.run(
    [
        sys.executable,
        "-m",
        "unittest",
        "tools.market_product_matching.test_pipeline",
        "-v",
    ],
    cwd=SOURCE_DIR,
    text=True,
    capture_output=True,
)
print(test_run.stdout)
print(test_run.stderr)
if test_run.returncode != 0:
    raise RuntimeError("Deterministic matcher tests failed; candidate generation has been stopped.")

from tools.market_product_matching.pipeline import (
    build_candidate_rows,
    build_mapping_payload,
    mapping_quality_report,
    read_review_csv,
    validate_catalogue_payload,
    write_candidates_csv,
    write_json,
)

market_products, market_listings, market_prices = validate_catalogue_payload(
    manifest,
    market_parts,
)
print("Market products:", len(market_products))
print("Retail listings:", len(market_listings))
print("Price observations:", len(market_prices))
print("Products needing food-scope review:", sum(bool(row.get("foodScopeReviewNeeded")) for row in market_products))



test_arabic_normalization_removes_diacritics_and_tatweel (tools.market_product_matching.test_pipeline.ProductIngredientPipelineTests.test_arabic_normalization_removes_diacritics_and_tatweel) ... ok
test_boiled_name_does_not_accidentally_trigger_oil_form (tools.market_product_matching.test_pipeline.ProductIngredientPipelineTests.test_boiled_name_does_not_accidentally_trigger_oil_form) ... ok
test_canned_product_does_not_match_a_generic_or_dried_ingredient (tools.market_product_matching.test_pipeline.ProductIngredientPipelineTests.test_canned_product_does_not_match_a_generic_or_dried_ingredient) ... ok
test_food_scope_review_products_cannot_be_approved (tools.market_product_matching.test_pipeline.ProductIngredientPipelineTests.test_food_scope_review_products_cannot_be_approved) ... ok
test_generated_candidates_remain_suggested_until_explicit_approval (tools.market_product_matching.test_pipeline.ProductIngredientPipelineTests.test_generated_candidates_remain_suggested_until_explicit_appr

## Generate the review queue

The engine compares product names and variants against canonical Arabic, French, and English food names/aliases. Candidate ranking is deterministic for the pinned files. It also blocks known form conflicts and incompatible unit families.

The review CSV contains up to five candidates per product. Review one product at a time; only the candidate you explicitly approve will become `VERIFIED`.

In [ ]:
MIN_CANDIDATE_SCORE = 42
MAX_CANDIDATES_PER_PRODUCT = 5

candidate_rows = build_candidate_rows(
    market_products,
    eligible_food_items,
    eligible_food_ids,
    min_score=MIN_CANDIDATE_SCORE,
    max_candidates=MAX_CANDIDATES_PER_PRODUCT,
)

quality_report = mapping_quality_report(
    market_products,
    candidate_rows,
    eligible_ingredient_count=len(eligible_food_items),
)
quality_report.update({
    "sourceCommit": SOURCE_COMMIT,
    "sourceCatalogueRevision": SOURCE_CATALOGUE_REVISION,
    "foodCatalogueRevision": FOOD_CATALOGUE_REVISION,
    "roomSchemaVersion": EXPECTED_ROOM_SCHEMA,
    "matchingAlgorithm": "midati-lexical-review-v1",
    "candidateScoreFloor": MIN_CANDIDATE_SCORE,
    "maxCandidatesPerProduct": MAX_CANDIDATES_PER_PRODUCT,
})

CANDIDATE_CSV = OUTPUT_DIR / "ingredient-match-review.csv"
write_candidates_csv(CANDIDATE_CSV, candidate_rows)
write_json(OUTPUT_DIR / "candidate-quality-report.json", quality_report)

print(json.dumps(quality_report, ensure_ascii=False, indent=2))
print("Review CSV:", CANDIDATE_CSV)
files.download(str(CANDIDATE_CSV))


## Review the CSV outside Colab

Open the downloaded CSV in Excel, LibreOffice, or Google Sheets.

For the one candidate you choose for a product, set `review_decision` to exactly:
- `APPROVE` — only when the retail item and canonical recipe ingredient are genuinely the same purchasable food.
- `REJECT` — when that candidate is wrong.
- Leave blank — not reviewed yet.

Do not delete, duplicate, or modify the generated candidate rows/IDs. If the second candidate is correct, approve that row instead of rank 1. A product may have **at most one** approved ingredient.

Products flagged for food-scope review cannot be approved in this pipeline. The exporter also rejects an incomplete or altered candidate list.

In [ ]:
uploaded = files.upload()
if uploaded:
    csv_names = [name for name in uploaded if name.lower().endswith(".csv")]
    if len(csv_names) != 1:
        raise ValueError("Upload exactly one edited review CSV.")
    REVIEW_CSV = OUTPUT_DIR / "uploaded-review.csv"
    REVIEW_CSV.write_bytes(uploaded[csv_names[0]])
    reviewed_rows = read_review_csv(REVIEW_CSV)
    print("Loaded reviewed rows:", len(reviewed_rows))
else:
    # Running this cell without an upload creates suggestions only.
    reviewed_rows = candidate_rows
    print("No reviewed CSV uploaded: export will contain SUGGESTED rows only.")


In [ ]:
generated_at_epoch_millis = int(datetime.now(timezone.utc).timestamp() * 1000)

mapping_payload = build_mapping_payload(
    market_products,
    candidate_rows,
    reviewed_rows,
    source_catalogue_revision=SOURCE_CATALOGUE_REVISION,
    generated_at_epoch_millis=generated_at_epoch_millis,
)

status_counts = Counter(row["status"] for row in mapping_payload["mappings"])
products_by_id = {str(row["id"]).lower(): row for row in market_products}
mapped_product_ids = [row["productId"] for row in mapping_payload["mappings"]]
if len(mapped_product_ids) != len(set(mapped_product_ids)):
    raise ValueError("More than one final mapping was generated for a product.")

for row in mapping_payload["mappings"]:
    if row["ingredientId"] not in eligible_food_ids:
        raise ValueError(f"Mapping references a non-eligible canonical ingredient: {row['ingredientId']}")
    if row["status"] == "VERIFIED":
        product = products_by_id[row["productId"]]
        if product.get("foodScopeReviewNeeded"):
            raise ValueError("A food-scope-review product was marked VERIFIED.")
        if row["matchMethod"] != "MANUAL" or not row["reviewedAtEpochMillis"]:
            raise ValueError("A VERIFIED mapping lacks manual review metadata.")

final_quality_report = dict(quality_report)
final_quality_report.update({
    "finalMappingRows": len(mapping_payload["mappings"]),
    "verifiedMappings": status_counts.get("VERIFIED", 0),
    "suggestedMappings": status_counts.get("SUGGESTED", 0),
    "rejectedMappings": status_counts.get("REJECTED", 0),
    "generatedAtEpochMillis": generated_at_epoch_millis,
    "sourceCatalogueRevision": SOURCE_CATALOGUE_REVISION,
    "foodCatalogueRevision": FOOD_CATALOGUE_REVISION,
})

MAPPING_JSON = OUTPUT_DIR / "ingredient-matches.json"
QA_JSON = OUTPUT_DIR / "mapping-qa-report.json"
SOURCE_METADATA_JSON = OUTPUT_DIR / "source-metadata.json"

write_json(MAPPING_JSON, mapping_payload)
write_json(QA_JSON, final_quality_report)
write_json(SOURCE_METADATA_JSON, {
    "repository": REPOSITORY,
    "sourceCommit": SOURCE_COMMIT,
    "sourceCatalogueRevision": SOURCE_CATALOGUE_REVISION,
    "foodCatalogueRevision": FOOD_CATALOGUE_REVISION,
    "roomSchemaVersion": EXPECTED_ROOM_SCHEMA,
    "marketProducts": len(market_products),
    "canonicalFoodItems": len(food_items),
    "eligibleRecipeAndSeedBackedIngredients": len(eligible_food_items),
    "crosswalkLegacyIds": len(legacy_to_food),
    "generatedAtEpochMillis": generated_at_epoch_millis,
})

BUNDLE = OUTPUT_DIR / "midati-product-ingredient-review-bundle.zip"
with zipfile.ZipFile(BUNDLE, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(CANDIDATE_CSV, arcname="ingredient-match-review.csv")
    archive.write(MAPPING_JSON, arcname="ingredient-matches.json")
    archive.write(QA_JSON, arcname="mapping-qa-report.json")
    archive.write(SOURCE_METADATA_JSON, arcname="source-metadata.json")

print("Final mapping status counts:", dict(status_counts))
print("Mapping JSON:", MAPPING_JSON)
print("QA report:", QA_JSON)
print("Bundle:", BUNDLE)
files.download(str(MAPPING_JSON))
files.download(str(BUNDLE))


## Import into the app repository

After review, place the exported `ingredient-matches.json` at:

`data/market-products/ingredient-matches.json`

Commit it on `main` alongside the generated QA report if desired. Do not edit `manifest.json` or the product part files as part of this mapping import.

The app computes a separate base-catalogue revision that intentionally excludes `ingredient-matches.json`; the mapping can therefore be refreshed independently while still being rejected when the underlying product catalogue changes.

The Android installer validates product IDs, ingredient IDs, source revision, and food-scope review status. On a fresh install it may defer the mapping until recipe ingredient rows have been installed; the repository retries installation on subsequent initialization. Only `VERIFIED` rows are eligible for the future product-backed price estimator. `SUGGESTED` and `REJECTED` rows are never price evidence.